In [25]:
import os
import pathlib

os.chdir(pathlib.Path().resolve())
print("Working directory set to:", os.getcwd())

Working directory set to: D:\github\Transport-Network-Design\Kinshasa\alg\unconstrained


<h3>Prerequisite for Collab</h3>

In [1]:
!pip install pyomo

Defaulting to user installation because normal site-packages is not writeable


In [2]:
!pip install highspy

Defaulting to user installation because normal site-packages is not writeable


In [3]:
!pip install pyscipopt

Defaulting to user installation because normal site-packages is not writeable


In [4]:
#!pip install maingopy

In [5]:
!pip install shapely

Defaulting to user installation because normal site-packages is not writeable


<h3>Prerequisite for Conda Installation</h3>

In [6]:
#On Mac Terminal: conda create -n optim_env python=3.10
#conda activate optim_env
#conda install -c conda-forge pyomo scip
#conda install -c conda-forge glpk     # Linear / MILP solver
#conda install -c conda-forge highs    # Linear / Quadratic solver
#conda install -c conda-forge ipopt    # Nonlinear solver
#conda install -c conda-forge scip     # MINLP solver
#conda install networkx
#conda install conda-forge::matplotlib

# **I. Network Analysis**

In [7]:
import networkx as nx
import matplotlib.pyplot as plt
import math
import time

# Step 1: Create graph
G2 = nx.Graph()

# Step 2: Define node data with BOTH position and label
# Each entry: node_id: {"pos": (x, y), "label": "..."}
node_data = {
    1:  {"pos": (-4.559058, 15.174809), "label": "Route de Matadi (border Kinshasa)"},
    2:  {"pos": (-4.442848, 15.255100), "label": "Triangle Matadi Kibala"},
    3:  {"pos": (-4.406924, 15.256513), "label": "UPN"},
    4:  {"pos": (-4.349325, 15.238810), "label": "Av. de l'École – Binza"},
    5:  {"pos": (-4.328773, 15.252333), "label": "Mont Ngaliema"},
    6:  {"pos": (-4.310907, 15.288488), "label": "Bd du 30 Juin"},
    7:  {"pos": (-4.299709, 15.319240), "label": "Gare Centrale"},
    8:  {"pos": (-4.398246, 15.276497), "label": "Selembao (Auto Stop)"},
    9:  {"pos": (-4.381511, 15.282728), "label": "Sanatorium"},
    10: {"pos": (-4.337062, 15.295951), "label": "Pierre Mulele"},
    11: {"pos": (-4.395099, 15.307741), "label": "Triangle Campus"},
    12: {"pos": (-4.389760, 15.314763), "label": "Rond Point Ngaba"},
    13: {"pos": (-4.336819, 15.326397), "label": "Av. de l'Université"},
    14: {"pos": (-4.331834, 15.314427), "label": "Bd Triomphal"},
    15: {"pos": (-4.393572, 15.333011), "label": "Lemba"},
    16: {"pos": (-4.375441, 15.344869), "label": "Échangeur 1"},
    17: {"pos": (-4.397829, 15.393514), "label": "Masina"},
    18: {"pos": (-4.405498, 15.423430), "label": "Av. Ndjoku"},
    19: {"pos": (-4.391380, 15.446782), "label": "Aéroport Ndjili"},
    20: {"pos": (-4.384103, 15.470908), "label": "Nsele"},
    21: {"pos": (-4.209434, 15.578420), "label": "RP Nsele"},
    22: {"pos": (-4.202280, 15.613246), "label": "Menkao"},
    23: {"pos": (-4.051282, 15.558907), "label": "Maluku"},
    # 23 intentionally omitted
    24: {"pos": (-4.472585, 15.281341), "label": "Arrêt Gare"},
    25: {"pos": (-4.420154, 15.307442), "label": "UNIKIN"},
    26: {"pos": (-4.461385, 15.288240), "label": "Kimwenza 2"},
    27: {"pos": (-4.484759, 15.219752), "label": "Benseke"},
    28: {"pos": (-4.447334, 15.249644), "label": "Wenze Matadi Kibala"},
    29: {"pos": (-4.329183, 15.337959), "label": "Limete PL"},
    30: {"pos": (-4.353936, 15.335540), "label": "Limete R"},
    
}

# Step 3: Add nodes with attributes
for node, attrs in node_data.items():
    G2.add_node(node, **attrs)  # adds both 'pos' and 'label'

# Step 4: Define edges with names
edges = [
    (1, 27, {"name": "Route de Matadi - Benseke"}),
    (27, 28, {"name": "Benseke - Wenze Matadi Kibala"}),
    (28, 2, {"name": "Wenze (Matadi Kibala) - Triangle (Cité Verte)"}),
    (2, 3, {"name": "Matadi Kibala - UPN"}),
    (3, 4, {"name": "UPN - Binza"}),
    (4, 5, {"name": "Binza - Mont Ngaliema"}),
    (5, 6, {"name": "Boulevard du 30 Juin"}),
    (6, 7, {"name": "24 - Gare Centrale"}),
    (3, 8, {"name": "UPN - Selembao"}),
    (8, 9, {"name": "Sanatorium - Université"}),
    (9, 10, {"name": "Sanatorium - Pierre Mulele"}),
    (10, 6, {"name": "24 Novembre"}),
    (2, 11, {"name": "Triangle (Cité Verte) - Triangle (Campus)"}),
    (11, 12, {"name": "Tr.Campus - RP Ngaba"}),
    (12, 13, {"name": "Av. Université"}),
    (10, 14, {"name": "B. Triomphal"}),
    (13, 14, {"name": "B. Sendwe"}),
    (12, 15, {"name": "RP Ngaba - Lemba"}),
    (15, 16, {"name": "Lemba - Echangeur 1"}),
    (13, 30, {"name": "Sendwe - Limete"}),
    (30, 16, {"name": "Limete. Lumumba 1"}),
    (16, 17, {"name": "Echangeur - Masina"}),
    (17, 18, {"name": "Masina - Av.Ndjoku (B.Lumumba 2)"}),
    (18, 19, {"name": "Av.Ndjoku - Int Aeroport (B.Lumumba 3)"}),#
    (19, 20, {"name": "Aeroport - Nsele"}),#
    (20, 21, {"name": "Nsele - RP. Nsele"}),#
    (21, 22, {"name": "RP.Nsele - Menkao"}),#
    (21, 23, {"name": "RP.Nsele - Maluku"}),#
    (2, 24, {"name": "Triangle Cité Verte - R. Kimwenza"}),#
    (11, 25, {"name": "Triangle Campus - Unikin"}),#
    (12, 25, {"name": "RP Ngaba - Unikin"}),#
    (25, 26, {"name": "Unikin - Kimwenza 2"}),#
    (7, 29, {"name": "Gare Centrale - Poids Lourd"}),#
    (29, 16, {"name": "Poids Lourd - Echangeur"}),#
]
G2.add_edges_from(edges)
G0 = G2.copy()

In [8]:
#(Distance) Calibration -------------------------------------
P1 = node_data[3]['pos']
P2 = node_data[27]['pos']
d = math.sqrt((P1[0]-P2[0])**2 + (P1[1]-P2[1])**2 )
True_d  = 4.2
calibration_score = d/True_d
print(d,calibration_score)
# ----------------------------------------------------------
# Edge lengths (Euclidean from node positions)
# ----------------------------------------------------------
def compute_edge_lengths(G):
    #pos = nx.get_node_attributes(G, 'pos')
    lengths = {}
    for (u, v) in G.edges(): 
        x1, y1 = node_data[u]['pos']
        x2, y2 = node_data[v]['pos']
        lengths[(u, v)] = ((x1 - x2) ** 2 + (y1 - y2) ** 2) ** 0.5
    return lengths

0.08607937236063044 0.02049508865729296


<h3>Problem Configuration</h3>

In [9]:
from joblib import load

demand_dict = load("../../data/od_demand_orig2.joblib")
print("Loaded", len(demand_dict), "OD pairs.")


Loaded 114 OD pairs.


In [22]:
#----- Complete Graph
Gc = nx.complete_graph(G0.nodes()) #complete graph
Gs = Gc.copy()
Gs.remove_edges_from(G0.edges())
complete_edge_list = Gs.edges()
#---- System Configuration ----------------------
lengths = compute_edge_lengths(Gc)
cost_weights = {(i,j): lengths[i,j] for (i,j) in Gc.edges()}
CALIB = calibration_score
#------------------------------------------------------ JSON --------------------
import json
with open("config.json", "r") as f:
    config = json.load(f)

Budget = config["Budget"]
v_avg = config["v_avg"]
penalty_factor = config["penalty_factor"]
optim_max_iter = config["optim_max_iter"]
optim_pop_size = config["optim_pop_size"]
n_runs = config["n_runs"]
folder = config["folder"]
C = config["C"]
#multiply demand
for key in demand_dict:
    demand_dict[key] = config["demand_scale"]*demand_dict[key]
#-------- Demand Data ---------------------------
OD_pairs = demand_dict.keys() #[(1,27),(16,18),(5,7),(10,13)] #Origin Destination Pairs
demand_paths = demand_dict# {(1,27):25, (16,18):15, (5,7):15, (10,13):18}
#----------------------------------------------------------------------------------
travel_t0 = {(i,j):(cost_weights[i,j]/calibration_score)/v_avg for (i,j) in Gc.edges()}
travel_alpha = {(i,j):0.15 for (i,j) in Gc.edges()}
#Optimisation settings
optim_stall_iter = optim_max_iter*optim_pop_size
#ACO
aco_alpha = 1.0
aco_beta = 1.0
aco_rho = 0.5
aco_Q = 1

FileNotFoundError: [Errno 2] No such file or directory: 'config.json'

In [23]:
import os
print(os.getcwd())

D:\github\Transport-Network-Design\Kinshasa\alg\unconstrained


<h3>Analysis</h3>

In [ ]:
import math
import networkx as nx
import matplotlib.pyplot as plt

def draw_rotated_graph(
    G,
    degrees=-90,
    figsize=(10, 8),
    invert_x=True,
    edge_flows=None,
    scale=1/200,
    show_labels=True,
    node_size=120,
    spread=2.0,
    label_pos=0.55,
    node_font_size=6,
    edge_font_size=6
):

    def rotate_pos(pos_dict, degrees):
        theta = math.radians(degrees)
        cos_t, sin_t = math.cos(theta), math.sin(theta)
        return {
            n: (x * cos_t + y * sin_t, -x * sin_t + y * cos_t)
            for n, (x, y) in pos_dict.items()
        }

    # Get positions
    pos = nx.get_node_attributes(G, 'pos')
    if not pos:
        pos = nx.spring_layout(G, seed=42, k=1.0)

    pos_rotated = rotate_pos(pos, degrees)
    pos_scaled = {n: (x*spread, y*spread) for n, (x,y) in pos_rotated.items()}

    # Edge widths
    edge_widths = []
    for (u,v) in G.edges():
        flow = 0.0
        if edge_flows:
            flow = edge_flows.get((u, v), edge_flows.get((v, u), 0.0))
        edge_widths.append(max(0.5, flow * scale))

    # Edge color normalization
    edge_values = []
    if edge_flows:
        flow_vals = [
            edge_flows.get((u,v), edge_flows.get((v,u), 0.0))
            for (u,v) in G.edges()
        ]
        f_min, f_max = min(flow_vals), max(flow_vals)

        for (u,v) in G.edges():
            f = edge_flows.get((u,v), edge_flows.get((v,u), 0.0))
            norm_f = (f - f_min) / (f_max - f_min) if f_max > f_min else 0.0
            edge_values.append(norm_f)
    else:
        edge_values = "grey"

    # Plot
    plt.figure(figsize=figsize)
    nx.draw_networkx_nodes(G, pos_scaled, node_size=node_size, node_color="lightblue")
    nx.draw_networkx_labels(G, pos_scaled, font_size=node_font_size, font_weight="bold")

    nx.draw_networkx_edges(
        G,
        pos_scaled,
        width=edge_widths,
        edge_color=edge_values,
        edge_cmap=plt.cm.RdYlGn_r,  # green=low, red=high
        arrows=G.is_directed()
    )

    if edge_flows and show_labels:
        edge_labels = {
            (u,v): f"{edge_flows.get((u,v), edge_flows.get((v,u), 0.0)):.2f}"
            for (u,v) in G.edges()
        }
        nx.draw_networkx_edge_labels(
            G,
            pos_scaled,
            edge_labels=edge_labels,
            font_size=edge_font_size,
            label_pos=label_pos,
            font_color="brown"
        )

    if invert_x:
        plt.gca().invert_xaxis()

    plt.axis("off")
    plt.show()


In [ ]:
draw_rotated_graph(G2)

In [ ]:
#Optimisation Preparation
import networkx as nx
Gs = Gc.copy()
Gs.remove_edges_from(G0.edges())
print('[0] Initial Network:',G0, '\n[1] Complete Network:', Gc,'\n[2] Solution Network:',Gs)
#candidate solution
Gi = G0
def get_simple_paths(Gi, r, s):
  return list(nx.all_simple_paths(Gi,source=r,target=s))

In [ ]:
from shapely.geometry import LineString
from shapely.strtree import STRtree
import itertools

def precompute_intersections(node_data):
    """
    Compute all intersecting edge pairs in a complete geometric graph.

    Parameters
    ----------
    node_data : dict
        {node_id: {"pos": (x, y)}}

    Returns
    -------
    list[tuple]
        [((u1, v1), (u2, v2)), ...] where each pair of edges intersects.
    """

    # === 1. Build edge list and geometries ===
    nodes = list(node_data.keys())
    edges = [(u, v) for u, v in itertools.combinations(nodes, 2)]
    geoms = [LineString([node_data[u]["pos"], node_data[v]["pos"]]) for u, v in edges]

    # === 2. Build spatial index ===
    tree = STRtree(geoms)

    # === 3. Map geometry id back to its index ===
    # These are actual geometry objects used in the tree.
    geom_to_index = {g: i for i, g in enumerate(geoms)}

    # === 4. Compute intersections ===
    intersection_pairs = set()

    for i, geom1 in enumerate(geoms):
        edge1 = edges[i]
        for geom2 in tree.query(geom1):
            # Some STRtree versions return proxy types; skip if not valid geometry
            if not isinstance(geom2, LineString):
                continue

            j = geom_to_index.get(geom2)
            if j is None or j == i:
                continue
            edge2 = edges[j]

            # Skip shared endpoints
            if set(edge1) & set(edge2):
                continue

            # Use .crosses() which ensures proper line intersection (not touching)
            if geom1.crosses(geom2):
                intersection_pairs.add(tuple(sorted([edge1, edge2])))

    return list(intersection_pairs)


In [ ]:
precomputed_pairs = precompute_intersections(node_data)
precomputed_pairs

In [ ]:
def count_intersections_subnetwork(G, precomputed_pairs):
    """
    Count how many of the precomputed intersection pairs exist in G.
    """
    edges_in_G = set(tuple(sorted(e)) for e in G.edges())
    count = sum(1 for (e1, e2) in precomputed_pairs
                if e1 in edges_in_G and e2 in edges_in_G)
    return count


In [ ]:
Gi_ = G0.copy()
Gi_.add_edges_from([(1,25),(1,6)])
draw_rotated_graph(Gi_)

In [ ]:
import pyomo.environ as pyo
from pyomo.opt import SolverStatus, TerminationCondition
import networkx as nx

# ---------------------------
# Helpers
# ---------------------------
def var2edge(edge_list, y_var):
    return [edge for edge, y_i in zip(edge_list, y_var) if y_i == 1]

def edge2var(edge_list, edges_):
    return [1 if (i, j) in edges_ or (j, i) in edges_ else 0 for (i, j) in edge_list]

def travel_time_discrete(t0, alpha, x_ij, C=1400.0):
    return t0 * (1 + alpha * (x_ij / C) ** 4)

# ---------------------------
# Helpers (undirected-safe)
# ---------------------------
def travel_time_integral(t0, alpha, x_ij, C=1400.0):
    # ∫ t0(1+α(x/C)^4) dx = t0(x + α*x^5/(5*C^4))
    return t0 * (x_ij + 0.2 * alpha * (x_ij**5) / (C**4))


def get_param_undirected(param_dict, i, j):
    if (i, j) in param_dict:
        return param_dict[(i, j)]
    if (j, i) in param_dict:
        return param_dict[(j, i)]
    raise KeyError(f"Missing parameter for edge {(i, j)} / {(j, i)}")


def configure_ipopt(solver, display=False):
    # Iterations / tolerances
    solver.options["max_iter"] = 5000
    solver.options["tol"] = 1e-7
    solver.options["acceptable_tol"] = 1e-6
    solver.options["acceptable_iter"] = 10

    # Robustness
    solver.options["mu_strategy"] = "adaptive"
    solver.options["nlp_scaling_method"] = "gradient-based"

    # Linear solver (depends on your IPOPT build)
    solver.options["linear_solver"] = "mumps"

    # Bound handling
    solver.options["bound_push"] = 1e-8
    solver.options["bound_frac"] = 1e-8

    # Output
    solver.options["print_level"] = 5 if display else 0
    return solver


# ---------------------------
# Lower Level Problem Solver (LLP) with DEBUG MARKERS
# ---------------------------
def LLP_problem_solve(
    G_0,
    design_edges,
    OD_pairs,
    demand_paths,
    travel_t0,
    travel_alpha,
    display=False,
    solver_="ipopt",
    k_paths=5,
    weight_attr="length",
    flow_upper_bound=None,   # if None, uses total OD demand
    keepfiles=False,
    logfile=None,
    # ---- debug controls ----
    debug_tag: str = "",
    debug_stop: bool = False,
    suspicious_Lopt_tol: float = 1e-12
):
    """
    Solves the LLP with IPOPT robustly, intended to be called many times in a greedy/TS loop.

    Debug markers:
      <<<LLP_DEBUG>>> NO_PATHS
      <<<LLP_DEBUG>>> ZERO_TOTAL_DEMAND
      <<<LLP_DEBUG>>> SOLVER_FAIL
      <<<LLP_DEBUG>>> SUSPICIOUS_L_OPT
    """

    # Build design network
    Gi = G_0.copy()
    Gi.add_edges_from(design_edges)

    # Build up to k shortest simple paths for each OD
    def get_flow_paths(G, r, s, k=k_paths, weight=weight_attr):
        gen = nx.shortest_simple_paths(G, source=r, target=s, weight=weight)
        paths_ = []
        for idx, path in enumerate(gen):
            if idx >= k:
                break
            paths_.append(list(zip(path[:-1], path[1:])))
        return paths_

    flow_paths = {(r, s): get_flow_paths(Gi, r, s) for (r, s) in OD_pairs}

    # --- DEBUG: disconnected OD pairs (no feasible path) ---
    disconnected = [(r, s) for (r, s), paths in flow_paths.items() if len(paths) == 0]
    if disconnected:
        msg = (
            f"<<<LLP_DEBUG>>> NO_PATHS | tag={debug_tag} | "
            f"|E_design|={len(design_edges)} | disconnected_ODs={len(disconnected)} | "
            f"examples={disconnected[:5]}"
        )
        print(msg)
        if debug_stop:
            raise RuntimeError(msg)
        return float("inf"), {}, {}

    # Create path-flow indices and dictionary mapping variable -> list of edges
    flow_dics = {}
    flow_vars_inds = []
    for (r, s), paths in flow_paths.items():
        for ii, path_edges in enumerate(paths, start=1):
            name = f"{(r, s)}_{ii}"
            flow_dics[name] = path_edges
            flow_vars_inds.append(name)

    # Compute stable upper bound for flows (prevents numerical explosion)
    total_demand = float(sum(float(demand_paths[(r, s)]) for (r, s) in OD_pairs))

    # --- DEBUG: total demand is zero (often a key mismatch bug) ---
    if total_demand <= 0:
        msg = (
            f"<<<LLP_DEBUG>>> ZERO_TOTAL_DEMAND | tag={debug_tag} | "
            f"total_demand={total_demand} | |OD_pairs|={len(OD_pairs)} | "
            f"examples_OD={OD_pairs[:5]}"
        )
        print(msg)
        if debug_stop:
            raise RuntimeError(msg)
        return float("inf"), {}, {}

    ub = float(flow_upper_bound) if flow_upper_bound is not None else total_demand
    if ub <= 0:
        # Extremely defensive: should not happen if total_demand>0
        ub = total_demand

    # Pyomo model
    m = pyo.ConcreteModel()

    # Decision variables with bounds
    m.x_ij = pyo.Var(Gi.edges(), within=pyo.NonNegativeReals, bounds=(0.0, ub))
    m.f_k = pyo.Var(flow_vars_inds, within=pyo.NonNegativeReals, bounds=(0.0, ub))

    # Demand satisfaction per OD: sum of path flows = demand
    def flow_demand_rule(model, r, s):
        return sum(
            model.f_k[f"{(r, s)}_{ii+1}"] for ii in range(len(flow_paths[(r, s)]))
        ) == float(demand_paths[(r, s)])

    m.constraint_flow = pyo.Constraint(OD_pairs, rule=flow_demand_rule)

    # Link-path consistency: x_ij equals sum of all path flows that use (i,j)
    def flow_xij_rule(model, i, j):
        expr = 0.0
        for key, path in flow_dics.items():
            if (i, j) in path or (j, i) in path:
                expr += model.f_k[key]
        return expr == model.x_ij[i, j]

    m.constraint_flow_xij = pyo.Constraint(Gi.edges(), rule=flow_xij_rule)

    # Objective: sum of integrals over edges
    m.obj = pyo.Objective(
        expr=sum(
            travel_time_integral(
                get_param_undirected(travel_t0, i, j),
                get_param_undirected(travel_alpha, i, j),
                m.x_ij[i, j],
            )
            for (i, j) in m.x_ij
        ),
        sense=pyo.minimize,
    )

    # Initialisation (helps IPOPT stability)
    for (i, j) in m.x_ij:
        m.x_ij[i, j].value = 0.0

    # Put each OD demand on the first path (feasible start in path space)
    for (r, s) in OD_pairs:
        dem = float(demand_paths[(r, s)])
        for ii in range(len(flow_paths[(r, s)])):
            name = f"{(r, s)}_{ii+1}"
            m.f_k[name].value = dem if ii == 0 else 0.0

    # Solve with robust failure handling
    solver = pyo.SolverFactory(solver_)
    if solver is None or (hasattr(solver, "available") and not solver.available()):
        raise RuntimeError(f"Solver '{solver_}' is not available in your environment.")

    if solver_ == "ipopt":
        solver = configure_ipopt(solver, display=display)

    solve_kw = dict(tee=display, load_solutions=False)
    if keepfiles:
        solve_kw["keepfiles"] = True
    if logfile is not None:
        solve_kw["logfile"] = logfile

    results = solver.solve(m, **solve_kw)

    ok_terms = {
        TerminationCondition.optimal,
        TerminationCondition.locallyOptimal,
        TerminationCondition.feasible,
    }

    # --- DEBUG: solver status/termination failure ---
    if (results.solver.status != SolverStatus.ok) or (results.solver.termination_condition not in ok_terms):
        msg = (
            f"<<<LLP_DEBUG>>> SOLVER_FAIL | tag={debug_tag} | "
            f"|E_design|={len(design_edges)} | total_demand={total_demand:.6g} | "
            f"status={results.solver.status} | term={results.solver.termination_condition}"
        )
        print(msg)
        if display:
            print("  message:", getattr(results.solver, "message", ""))
        if debug_stop:
            raise RuntimeError(msg)
        return float("inf"), {}, {}

    # Load and extract solution
    m.solutions.load_from(results)

    x_flows = {(i, j): float(pyo.value(m.x_ij[i, j])) for (i, j) in m.x_ij}
    f_flows = {k: float(pyo.value(m.f_k[k])) for k in m.f_k}
    L_opt = float(pyo.value(m.obj))

    # --- DEBUG: suspiciously tiny objective (common when demand accidentally near 0) ---
    if L_opt < suspicious_Lopt_tol:
        msg = (
            f"<<<LLP_DEBUG>>> SUSPICIOUS_L_OPT | tag={debug_tag} | "
            f"L_opt={L_opt:.3e} | total_demand={total_demand:.6g} | "
            f"|E_design|={len(design_edges)} | |x|={len(m.x_ij)}"
        )
        print(msg)
        if debug_stop:
            raise RuntimeError(msg)

    if display:
        print(f"Number of x-variables: {len(m.x_ij)}")
        for (i, j), val in list(x_flows.items())[:20]:
            print(f"x[{i},{j}] = {val}")
        if len(x_flows) > 20:
            print("... (truncated)")

        print(f"Number of f-variables: {len(m.f_k)}")
        for k, val in list(f_flows.items())[:20]:
            print(f"f[{k}] = {val}")
        if len(f_flows) > 20:
            print("... (truncated)")

        print("Objective (L_opt):", L_opt)

    return L_opt, x_flows, f_flows

In [ ]:
import pyomo.environ as pyo
from pyomo.opt import SolverStatus, TerminationCondition
import networkx as nx

# ---------------------------
# Helpers
# ---------------------------
def var2edge(edge_list, y_var):
    return [edge for edge, y_i in zip(edge_list, y_var) if y_i == 1]

def edge2var(edge_list, edges_):
    return [1 if (i, j) in edges_ or (j, i) in edges_ else 0 for (i, j) in edge_list]

def travel_time_discrete(t0, alpha, x_ij, C=2800.0):
    return t0 * (1 + alpha * (x_ij / C) ** 4)

# ---------------------------
# Helpers (undirected-safe)
# ---------------------------
def travel_time_integral(t0, alpha, x_ij, C=2800.0):
    # ∫ t0(1+α(x/C)^4) dx = t0(x + α*x^5/(5*C^4))
    return t0 * (x_ij + 0.2 * alpha * (x_ij**5) / (C**4))


def get_param_undirected(param_dict, i, j):
    if (i, j) in param_dict:
        return param_dict[(i, j)]
    if (j, i) in param_dict:
        return param_dict[(j, i)]
    raise KeyError(f"Missing parameter for edge {(i, j)} / {(j, i)}")


def configure_ipopt(solver, display=False):
    # Iterations / tolerances
    solver.options["max_iter"] = 5000
    solver.options["tol"] = 1e-7
    solver.options["acceptable_tol"] = 1e-6
    solver.options["acceptable_iter"] = 10

    # Robustness
    solver.options["mu_strategy"] = "adaptive"
    solver.options["nlp_scaling_method"] = "gradient-based"

    # Linear solver (depends on your IPOPT build)
    solver.options["linear_solver"] = "mumps"

    # Bound handling
    solver.options["bound_push"] = 1e-8
    solver.options["bound_frac"] = 1e-8

    # Output
    solver.options["print_level"] = 5 if display else 0
    return solver


# ---------------------------
# Lower Level Problem Solver (LLP) with DEBUG MARKERS
# ---------------------------
def LLP_problem_solve(
    G_0,
    design_edges,
    OD_pairs,
    demand_paths,
    travel_t0,
    travel_alpha,
    display=False,
    solver_="ipopt",
    k_paths=5,
    weight_attr="length",
    flow_upper_bound=None,   # if None, uses total OD demand
    keepfiles=False,
    logfile=None,
    # ---- debug controls ----
    debug_tag: str = "",
    debug_stop: bool = False,
    suspicious_Lopt_tol: float = 1e-12
):
    """
    Solves the LLP with IPOPT robustly, intended to be called many times in a greedy/TS loop.

    Debug markers:
      <<<LLP_DEBUG>>> NO_PATHS
      <<<LLP_DEBUG>>> ZERO_TOTAL_DEMAND
      <<<LLP_DEBUG>>> SOLVER_FAIL
      <<<LLP_DEBUG>>> SUSPICIOUS_L_OPT
    """

    # Build design network
    Gi = G_0.copy()
    Gi.add_edges_from(design_edges)

    # Build up to k shortest simple paths for each OD
    def get_flow_paths(G, r, s, k=k_paths, weight=weight_attr):
        gen = nx.shortest_simple_paths(G, source=r, target=s, weight=weight)
        paths_ = []
        for idx, path in enumerate(gen):
            if idx >= k:
                break
            paths_.append(list(zip(path[:-1], path[1:])))
        return paths_

    flow_paths = {(r, s): get_flow_paths(Gi, r, s) for (r, s) in OD_pairs}

    # --- DEBUG: disconnected OD pairs (no feasible path) ---
    disconnected = [(r, s) for (r, s), paths in flow_paths.items() if len(paths) == 0]
    if disconnected:
        msg = (
            f"<<<LLP_DEBUG>>> NO_PATHS | tag={debug_tag} | "
            f"|E_design|={len(design_edges)} | disconnected_ODs={len(disconnected)} | "
            f"examples={disconnected[:5]}"
        )
        print(msg)
        if debug_stop:
            raise RuntimeError(msg)
        return float("inf"), {}, {}

    # Create path-flow indices and dictionary mapping variable -> list of edges
    flow_dics = {}
    flow_vars_inds = []
    for (r, s), paths in flow_paths.items():
        for ii, path_edges in enumerate(paths, start=1):
            name = f"{(r, s)}_{ii}"
            flow_dics[name] = path_edges
            flow_vars_inds.append(name)

    # Compute stable upper bound for flows (prevents numerical explosion)
    total_demand = float(sum(float(demand_paths[(r, s)]) for (r, s) in OD_pairs))

    # --- DEBUG: total demand is zero (often a key mismatch bug) ---
    if total_demand <= 0:
        msg = (
            f"<<<LLP_DEBUG>>> ZERO_TOTAL_DEMAND | tag={debug_tag} | "
            f"total_demand={total_demand} | |OD_pairs|={len(OD_pairs)} | "
            f"examples_OD={OD_pairs[:5]}"
        )
        print(msg)
        if debug_stop:
            raise RuntimeError(msg)
        return float("inf"), {}, {}

    ub = float(flow_upper_bound) if flow_upper_bound is not None else total_demand
    if ub <= 0:
        # Extremely defensive: should not happen if total_demand>0
        ub = total_demand

    # Pyomo model
    m = pyo.ConcreteModel()

    # Decision variables with bounds
    m.x_ij = pyo.Var(Gi.edges(), within=pyo.NonNegativeReals, bounds=(0.0, ub))
    m.f_k = pyo.Var(flow_vars_inds, within=pyo.NonNegativeReals, bounds=(0.0, ub))

    # Demand satisfaction per OD: sum of path flows = demand
    def flow_demand_rule(model, r, s):
        return sum(
            model.f_k[f"{(r, s)}_{ii+1}"] for ii in range(len(flow_paths[(r, s)]))
        ) == float(demand_paths[(r, s)])

    m.constraint_flow = pyo.Constraint(OD_pairs, rule=flow_demand_rule)

    # Link-path consistency: x_ij equals sum of all path flows that use (i,j)
    def flow_xij_rule(model, i, j):
        expr = 0.0
        for key, path in flow_dics.items():
            if (i, j) in path or (j, i) in path:
                expr += model.f_k[key]
        return expr == model.x_ij[i, j]

    m.constraint_flow_xij = pyo.Constraint(Gi.edges(), rule=flow_xij_rule)

    # Objective: sum of integrals over edges
    m.obj = pyo.Objective(
        expr=sum(
            travel_time_integral(
                get_param_undirected(travel_t0, i, j),
                get_param_undirected(travel_alpha, i, j),
                m.x_ij[i, j],
            )
            for (i, j) in m.x_ij
        ),
        sense=pyo.minimize,
    )

    # Initialisation (helps IPOPT stability)
    for (i, j) in m.x_ij:
        m.x_ij[i, j].value = 0.0

    # Put each OD demand on the first path (feasible start in path space)
    for (r, s) in OD_pairs:
        dem = float(demand_paths[(r, s)])
        for ii in range(len(flow_paths[(r, s)])):
            name = f"{(r, s)}_{ii+1}"
            m.f_k[name].value = dem if ii == 0 else 0.0

    # Solve with robust failure handling
    solver = pyo.SolverFactory(solver_)
    if solver is None or (hasattr(solver, "available") and not solver.available()):
        raise RuntimeError(f"Solver '{solver_}' is not available in your environment.")

    if solver_ == "ipopt":
        solver = configure_ipopt(solver, display=display)

    solve_kw = dict(tee=display, load_solutions=False)
    if keepfiles:
        solve_kw["keepfiles"] = True
    if logfile is not None:
        solve_kw["logfile"] = logfile

    results = solver.solve(m, **solve_kw)

    ok_terms = {
        TerminationCondition.optimal,
        TerminationCondition.locallyOptimal,
        TerminationCondition.feasible,
    }

    # --- DEBUG: solver status/termination failure ---
    if (results.solver.status != SolverStatus.ok) or (results.solver.termination_condition not in ok_terms):
        msg = (
            f"<<<LLP_DEBUG>>> SOLVER_FAIL | tag={debug_tag} | "
            f"|E_design|={len(design_edges)} | total_demand={total_demand:.6g} | "
            f"status={results.solver.status} | term={results.solver.termination_condition}"
        )
        print(msg)
        if display:
            print("  message:", getattr(results.solver, "message", ""))
        if debug_stop:
            raise RuntimeError(msg)
        return float("inf"), {}, {}

    # Load and extract solution
    m.solutions.load_from(results)

    x_flows = {(i, j): float(pyo.value(m.x_ij[i, j])) for (i, j) in m.x_ij}
    f_flows = {k: float(pyo.value(m.f_k[k])) for k in m.f_k}
    L_opt = float(pyo.value(m.obj))

    # --- DEBUG: suspiciously tiny objective (common when demand accidentally near 0) ---
    if L_opt < suspicious_Lopt_tol:
        msg = (
            f"<<<LLP_DEBUG>>> SUSPICIOUS_L_OPT | tag={debug_tag} | "
            f"L_opt={L_opt:.3e} | total_demand={total_demand:.6g} | "
            f"|E_design|={len(design_edges)} | |x|={len(m.x_ij)}"
        )
        print(msg)
        if debug_stop:
            raise RuntimeError(msg)

    if display:
        print(f"Number of x-variables: {len(m.x_ij)}")
        for (i, j), val in list(x_flows.items())[:20]:
            print(f"x[{i},{j}] = {val}")
        if len(x_flows) > 20:
            print("... (truncated)")

        print(f"Number of f-variables: {len(m.f_k)}")
        for k, val in list(f_flows.items())[:20]:
            print(f"f[{k}] = {val}")
        if len(f_flows) > 20:
            print("... (truncated)")

        print("Objective (L_opt):", L_opt)

    return L_opt, x_flows, f_flows

<h2>Ant Colony Optimisation</h2>

In [ ]:
import numpy as np
import random
import copy
import time

def aco_upper_solver(G0,
                     complete_edge_list,
                     OD_pairs,
                     demand_paths,
                     travel_t0,
                     travel_alpha,
                     cost_weights,
                     budget=100,
                     calib=1.0,
                     penalty_factor=1e9,
                     alpha=aco_alpha,
                     beta=aco_beta,
                     rho=aco_rho,
                     Q=aco_Q,
                     num_ants=10,
                     max_iter=50,
                     display=False):
    """
    Ant Colony Optimisation (ACO) solver for the Upper Level Problem (ULP).
    
    Parameters
    ----------
    G0 : networkx.Graph
        Base graph.
    complete_edge_list : list
        All possible edges (tuples).
    OD_pairs, demand_paths : dict
        Precomputed OD data.
    travel_t0, travel_alpha : dict
        Travel time parameters.
    cost_weights : dict
        Edge construction costs.
    budget : float
        Total available budget.
    calib : float
        Calibration factor for budget scaling.
    penalty_factor : float
        Penalty for infeasible solutions.
    alpha, beta : float
        Pheromone and heuristic influence.
    rho : float
        Pheromone evaporation rate.
    Q : float
        Pheromone deposit constant.
    num_ants : int
        Number of ants per iteration.
    max_iter : int
        Maximum iterations.
    display : bool
        Whether to display progress.
    """

    # === INITIALISE PHEROMONES AND HEURISTIC INFORMATION ===
    tau = {e: 1.0 for e in complete_edge_list}  # pheromone level
    eta = {e: 1.0 / (cost_weights[e] + 1e-9) for e in complete_edge_list}  # heuristic info

    best_edges, best_obj, best_flows = None, float("inf"), None

    iter_history = {
        "edges": [],
        "obj": [],
        "flows": []
    }

    for it in range(max_iter):
        all_solutions = []
        all_objs = []
        all_flows = []

        # === Each ant constructs a candidate network ===
        for ant in range(num_ants):
            E_s = []
            total_cost = 0.0

            while total_cost < budget * calib:
                # Candidate edges not yet added
                remaining_edges = [e for e in complete_edge_list
                                   if e not in E_s and e not in G0.edges()]

                # Filter feasible edges (respect budget)
                feas_edges = [e for e in remaining_edges
                              if total_cost + cost_weights[e] <= budget * calib]
                if not feas_edges:
                    break

                # --- Compute probability distribution P(e) ---
                pheromone_scores = np.array([tau[e] ** alpha * eta[e] ** beta for e in feas_edges])
                total_pheromone = pheromone_scores.sum()
                if total_pheromone <= 0:
                    # Avoid division by zero
                    P = np.ones(len(feas_edges)) / len(feas_edges)
                else:
                    P = pheromone_scores / total_pheromone

                # --- Roulette-wheel selection (FIX: choose index) ---
                idx = np.random.choice(len(feas_edges), p=P)
                next_edge = feas_edges[idx]

                # --- Add edge and update total cost ---
                E_s.append(next_edge)
                total_cost += cost_weights[next_edge]

            # === Evaluate ant's constructed network ===
            y_trial = [1 if edge in E_s else 0 for edge in complete_edge_list]

            obj_val, flows, pure_cost = Upper_Level_Problem_obj_func(
                G0,
                y_vars=y_trial,
                OD_pairs=OD_pairs,
                demand_paths=demand_paths,
                travel_t0=travel_t0,
                travel_alpha=travel_alpha,
                cost_weights=cost_weights,
                budget=budget,
                penalty_factor=penalty_factor,
                calib=calib,
                complete_edge_vars=complete_edge_list,
                display=False
            )

            # Store results
            all_solutions.append(E_s)
            all_objs.append(obj_val)
            all_flows.append(flows)

            # Track global best
            if obj_val < best_obj:
                best_obj = obj_val
                best_edges = E_s.copy()
                best_flows = flows

        # === PHEROMONE UPDATE ===
        # Evaporation
        for e in tau:
            tau[e] = (1 - rho) * tau[e]

        # Deposit pheromone based on solution quality
        for i, E_s in enumerate(all_solutions):
            delta_tau = Q / (all_objs[i] + 1e-9)
            for e in E_s:
                tau[e] += delta_tau


        iter_history["edges"].append(best_edges.copy())
        iter_history["flows"].append(best_flows.copy())
        iter_history["obj"].append(best_obj)

        # === Display progress ===
        if display:
            print(f"[ACO] Iter {it+1:03d}/{max_iter}: "
                  f"BestObj={best_obj:.4f}, MeanObj={np.mean(all_objs):.4f}")

            # Optional graph drawing
            Gi = G0.copy()
            if best_edges:
                Gi.add_edges_from(best_edges)
            """
            try:
                draw_rotated_graph(Gi, edge_flows=best_flows)
            except Exception:
                pass
            """

    # === Return final best ===
    return best_edges, best_obj, best_flows, iter_history


In [ ]:

aco_best_edges_list   = []
aco_best_obj_list     = []
aco_best_flows_list   = []
aco_iter_history_list = []
elapsed_aco_list      = []

start_aco = time.time()

aco_best_edges, aco_best_obj, aco_best_flows, aco_iter_history = aco_upper_solver(
    G0,
    complete_edge_list,
    OD_pairs=OD_pairs,
    demand_paths=demand_paths,
    travel_t0=travel_t0,
    travel_alpha=travel_alpha,
    cost_weights=cost_weights,
    budget=Budget,
    calib=CALIB,
    num_ants=optim_pop_size,
    max_iter=optim_max_iter,
    alpha=aco_alpha,
    beta=aco_beta,
    rho=aco_rho,
    Q=aco_Q,
    display=True
)

stop_aco = time.time()
elapsed_aco = (stop_aco - start_aco)/60

# --- Store results from this run ---
elapsed_aco_list.append(elapsed_aco)
aco_best_edges_list.append(aco_best_edges.copy())
aco_best_obj_list.append(aco_best_obj)
aco_best_flows_list.append(aco_best_flows.copy())
aco_iter_history_list.append(aco_iter_history.copy())



In [ ]:
print("\nACO Chosen edges:", aco_best_edges)
print("Final objective:", aco_best_obj)
print(f"Elapsed time: {elapsed_aco:.4f} minutes")

<h3>ACO Multiple Runs</h3>

In [ ]:
# ===============================================================
# === MULTIPLE RUNS: ANT COLONY OPTIMISATION ====================
# ===============================================================


for i in range(n_runs):
    print(f'ACO run number {i+1}...')
    # ------------------------------------------------------------
    start_aco = time.time()
    
    aco_best_edges, aco_best_obj, aco_best_flows, aco_iter_history = aco_upper_solver(
        G0,
        complete_edge_list,
        OD_pairs=OD_pairs,
        demand_paths=demand_paths,
        travel_t0=travel_t0,
        travel_alpha=travel_alpha,
        cost_weights=cost_weights,
        budget=Budget,
        calib=CALIB,
        num_ants=optim_pop_size,
        max_iter=optim_max_iter,
        alpha=aco_alpha,
        beta=aco_beta,
        rho=aco_rho,
        Q=aco_Q,
        display=False
    )
    
    stop_aco = time.time()
    elapsed_aco = (stop_aco - start_aco) / 60.0  # minutes
    
    # --- Store results from this run ---
    elapsed_aco_list.append(elapsed_aco)
    aco_best_edges_list.append(aco_best_edges.copy())
    aco_best_obj_list.append(aco_best_obj)
    aco_best_flows_list.append(aco_best_flows.copy())
    aco_iter_history_list.append(aco_iter_history.copy())
    
    # --- Display per-run summary ---
    print("\n=== ACO Search Result ===")
    print("Best edges selected:", aco_best_edges)
    print("Best objective value:", aco_best_obj)
    print(f"ACO Elapsed Time: {elapsed_aco:.4f} minutes\n")

print("ACO search multiple runs complete...")


<h2>Saving Results</h2>

In [ ]:
from joblib import dump, load
from datetime import datetime

timestamp = datetime.now().strftime("%d_%m_%Y_%H_%M_%S")
aco_title = f"../../output/results/unconstrained/{folder}/aco_results_{timestamp}.joblib"

In [ ]:
# ===============================================================
# === SAVE ACO RESULTS ==========================================
# ===============================================================

aco_solution_pack = {
    "optim_name": "ACO",
    "desc": "no intersection to objective function",
    "n_sol": n_runs + 1,
    "optim_time": elapsed_aco_list,
    "settings": [{
        "num_ants": optim_pop_size,
        "max_iter": optim_max_iter,
        "alpha": aco_alpha,
        "beta": aco_beta,
        "rho": aco_rho,
        "Q": aco_Q
    }],
    "solutions": aco_best_edges_list,
    "obj_vals": aco_best_obj_list,
    "flows": aco_best_flows_list,
    "iter_history": aco_iter_history_list
}



dump(aco_solution_pack, aco_title)
print(f"{aco_title} saved....")

In [ ]:
print("\n=== ACO MULTI-RUN SUMMARY ===")
print(f"Average objective: {np.mean(aco_best_obj_list):.4f}")
print(f"Best objective: {np.min(aco_best_obj_list):.4f}")
print(f"Average runtime: {np.mean(elapsed_aco_list):.4f} minutes")
print(f"Std. deviation of objective: {np.std(aco_best_obj_list):.4f}")